In [8]:
import warnings
import pynamod
import torch
import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from MDAnalysis.analysis import polymer
warnings.filterwarnings('ignore')

In [2]:
# insert filepaths here
trajectory_file = '../../test_traj.h5'
structure_file = '../../test_cg.h5'

In [3]:
def get_cos(cgs):
    coords = []
    ref_inds = np.array([protein.ref_pair.ind for protein in cgs.proteins])
    ref_vecs = cgs.dna.origins[ref_inds].reshape(-1,3)
    ref_vecs = np.diff(ref_vecs,axis=0)
    ref_vecs /= np.linalg.norm(ref_vecs,axis=1,keepdims=True)
    for st in cgs.dna.trajectory:
        coords.append(cgs.dna.origins[ref_inds].to(torch.double).reshape(-1,3).numpy())
    coords = np.stack([coords])[0]    
    test_vecs = coords[:,1:] - coords[:,:-1]
    test_vecs/=np.expand_dims(np.linalg.norm(test_vecs,axis=2),2)
    cosines = np.einsum('kij,kij->ki',ref_vecs.reshape(-1,*ref_vecs.shape),test_vecs )
    return cosines


def get_gyration_radii(cgs,ln):
    pair_names = [pair.pair_name.replace("B'",'') for pair in cgs.dna.pairs_list]
    total_masses = np.hstack([protein.masses for protein in cgs.proteins])
    M = total_masses.sum()
    gyr_radii = np.zeros(ln)
    prot_vectors = cgs.proteins[0].ref_vectors.reshape(-1,3).numpy()
    ref_inds = np.array([protein.ref_pair.ind for protein in cgs.proteins])

    for i,st in enumerate(cgs.dna.trajectory):
        ref_r = cgs.dna.ref_frames[ref_inds].numpy().transpose(0,2,1)
        pos = cgs.origins.numpy()
        center = np.sum(pos*total_masses.reshape(-1,1),axis=0)/M
        sm = np.sum((pos - center)**2,axis=1)
        radii_square = np.sum(total_masses.reshape(-1,1)*sm) / M
        if i < ln:
            gyr_radii[i] = np.sqrt(radii_square)

    return gyr_radii


def get_end_end_dist(cgs, ln):
    dist = np.zeros(ln)
    ind1 = [p.ref_pair.ind for p in cgs.proteins[6:9]]
    ind2 = [p.ref_pair.ind for p in cgs.proteins[13:10:-1]]
    for i,st in enumerate(cgs.dna.trajectory):
        ori = cgs.dna.origins.to(torch.double).numpy()
        if i < ln:
            dist[i] = np.linalg.norm(ori[ind1]-ori[ind2],axis=1).mean()

    return dist

def get_central_dist(cgs, ln):
    dist = np.zeros(ln)
    ind1 = [p.ref_pair.ind for p in cgs.proteins[6:9]]
    ind2 = [p.ref_pair.ind for p in cgs.proteins[13:10:-1]]
    for i,st in enumerate(cgs.dna.trajectory):
        ori = cgs.dna.origins.to(torch.double).numpy()
        if i < ln:
            
            dist[i] = np.linalg.norm(ori[ind1]-ori[ind2],axis=1).mean()


def analyze_traj(cgs, traj_file, step):
    cgs.dna.traj_step = step
    ln = len(cgs.dna.geom_params.trajectory)//step

    cosines = get_cos(cgs)
    gyration_radii = get_gyration_radii(cgs, ln)
    dist = get_end_end_dist(cgs, ln)
    central_dist = get_central_dist(cgs, ln)

    return cosines, gyration_radii, dist,central_dist

In [4]:
cgs = pynamod.CG_Structure()
cgs.load_from_h5(h5py.File(structure_file, 'r'))
cgs.dna.transfer_trajectory_to_h5(trajectory_file, 'r')

In [6]:
step = 10
#proteins in the bodies of functions get_central_dist and get_end_end_dist need fixing
cosines, gyration_radii, dist,central_dist = analyze_traj(cgs, trajectory_file, step)

In [ ]:
cgs.dna.traj_step = 10
u = cgs.get_cg_mda_traj()

ref_inds = np.array([protein.ref_pair.ind for protein in cgs.proteins if protein.n_cg_beads > 10])
backbones = u.atoms[ref_inds]


In [9]:
plen = polymer.PersistenceLength([backbones])
plen.run()
plen.results.lp

np.float64(1.0)

In [10]:
def exp_func(x, lp):
    return np.exp(-x/lp)

In [ ]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)

val = cosines.mean(axis=0)
popt, pcov = curve_fit(exp_func, np.arange(len(val)), val)
ax.plot(val)
ax.plot(exp_func(np.arange(len(val)),*popt),'--')

plt.xticks(range(1,19,2))
plt.xlabel('число нуклеосом')
plt.ylabel('ориентационная корреляция')

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)

ax.plot(np.arange(0,gyration_radii.shape[0],10,int)/10,gyration_radii)

plt.xlabel('Кадры, тысячи')
plt.ylabel('радиус гирации, Å')

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)

val,edges=np.histogram(gyration_radii,bins=50,density=True)
ax.plot(edges[1:])

plt.ylabel('Частота встречаемости')
plt.xlabel('Радиус гирации')

In [ ]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)
y = 17.5
def Hooke_law(x, K, b, c):
    return (0.5 * K * (x-b)**2)+c

    
val,edges=np.histogram(central_dist,bins=50,density=True)
E = -k*300*N_A*0.001*np.log(val)
popt, pcov = curve_fit(Hooke_law, edges, E,p0=init)

ax.plot(edges,E-popt[-1],'.')
ax.plot(edges,Hooke_law(edges,*popt)-popt[-1],'--',color=color)
plt.ylabel('E, КДж/моль')
plt.xlabel('R, Å')